In [ ]:
#imports
import pandas as pd
import numpy as np

In [ ]:
#analyze initial wide parquet of metabolites
df = pd.read_parquet("/content/wide_metabolites.parquet")
severity = df["depression_status"].astype(int)
y = (severity >= 2).astype(int)
meta_cols = ["depression_status", "file_name"]
X = df.drop(columns=meta_cols)
X = X.apply(pd.to_numeric, errors="coerce")

In [ ]:
#import hmdb_name_to_id csv and clean it
A = pd.read_csv("/content/hmdb_name_to_id.csv")
def clean(s):
  return (s.astype(str).str.strip().str.lower().str.replace(r"\s+", " ", regex=True))
A["query_name_clean"] = clean(A["query_name"])

In [ ]:
#apply the cleaning changes to the csv
X_cols = pd.Series(X.columns, name="orig_name")
X_map = pd.DataFrame({
  "orig_name": X.columns,
  "clean": clean(pd.Series(X.columns))
})

In [ ]:
#match original targets to the cleaned dataset
targets = set(A["query_name_clean"])
matched = X_map[X_map["clean"].isin(targets)]
print("Matched columns:", len(matched))

Matched columns: 312


In [ ]:
#print shape
X_312 = X[matched["orig_name"]].copy()
print(X_312.shape)

(269, 312)


In [ ]:
#match the ids to the hmdb csv
name_to_hmdb = dict(zip(A["query_name_clean"], A["hmdb_id"]))
matched = matched.copy()
matched.loc[:, "hmdb_id"] = matched["clean"].map(name_to_hmdb)
# rename columns from names → HMDB IDs
rename_dict = dict(zip(matched["orig_name"], matched["hmdb_id"]))
X_312 = X_312.rename(columns=rename_dict)

In [ ]:
#standardize metabolites
X = X_312.astype(float)
X_z = (X - X.mean(axis=0)) / X.std(axis=0, ddof=0)
#handle numerical instability after scaling
X_z = X_z.replace([np.inf, -np.inf], np.nan).fillna(0)
#compute mean standardized intensity per metabolite and save as csv
y_series = pd.Series(y, index=X_z.index)
group_means = X_z.groupby(y_series).mean()
B = group_means.T.reset_index()
B.columns = ["metabolite", "Control", "Depressed"]
B.to_csv("intensity_by_group.csv", index=False)

In [ ]:
B

,metabolite,Control,Depressed
0,HMDB0059726,0.035375,-0.045958
1,HMDB0003375,0.035375,-0.045958
2,HMDB0036076,0.035375,-0.045958
3,HMDB0034697,0.035375,-0.045958
4,HMDB0006525,0.035375,-0.045958
...,...,...,...
307,HMDB0038150,0.035375,-0.045958
308,HMDB0010729,-0.078189,0.101579
309,HMDB0030089,0.035375,-0.045958
310,HMDB0037742,-0.078189,0.101579


In [ ]:
#import hmdb final dataset
tax = pd.read_csv("/content/hmdb_final_dataset.csv")
tax.head()

,query_name,hmdb_id,status,http_url,best_match_name,match_score,n_hits,kingdom,superclass,class,subclass,process_terms
0,"2,6-Dimethyl-5-heptenal propyleneglycol acetal",HMDB0032235,404_not_found,http://35.184.189.38/api/hmdb/metabolites/sear...,NaN,NaN,NaN,Organic compounds,Organoheterocyclic compounds,Dioxolanes,"1,3-dioxolanes",NaN
1,DG(i-12:0/i-15:0/0:0),HMDB0093061,ok,NaN,DG(i-12:0/i-15:0/0:0),100.0,1.0,Organic compounds,Lipids and lipid-like molecules,Glycerolipids,Diradylglycerols,NaN
2,(±)-Citronellyl acetate,HMDB0034160,404_not_found,http://35.184.189.38/api/hmdb/metabolites/sear...,NaN,NaN,NaN,Organic compounds,Lipids and lipid-like molecules,Fatty Acyls,Fatty alcohol esters,NaN
3,PA(18:1(11Z)/18:0),HMDB0114901,ok,NaN,PA(18:1(11Z)/18:0),100.0,1.0,Organic compounds,Lipids and lipid-like molecules,Glycerophospholipids,Glycerophosphates,NaN
4,2-Hexenyl hexanoate,HMDB0038924,404_not_found,http://35.184.189.38/api/hmdb/metabolites/sear...,NaN,NaN,NaN,Organic compounds,Lipids and lipid-like molecules,Fatty Acyls,Fatty acid esters,NaN


In [ ]:
# normalize HMDB formatting to guarantee merge hits
def norm_hmdb(s):
  s = s.astype(str).str.strip().str.upper()
  s = s.str.extract(r"(HMDB\d+)", expand=False)
  return s

#normalize HMDB IDs in both datasets
B = pd.read_csv("intensity_by_group.csv")
B["hmdb_id_norm"] = norm_hmdb(B["hmdb_id"])
tax["hmdb_id_norm"] = norm_hmdb(tax["hmdb_id"])

#merge metabolite table with taxonomy
cat = (B[["hmdb_id_norm"]].merge(tax[["hmdb_id_norm","class"]], on="hmdb_id_norm", how="left"))
cat = cat.rename(columns={"hmdb_id_norm":"hmdb_id"})
cat["class"] = cat["class"].fillna("Unknown")

#export final metabolite-to-class mapping
cat.to_csv("metabolite_category.csv", index=False)

#print dataset size and most frequent classes
print(cat.shape)
print(cat["class"].value_counts().head(10))

(313, 2)
class
Glycerolipids                          91
Prenol lipids                          45
Fatty Acyls                            44
Glycerophospholipids                   39
Carboxylic acids and derivatives       26
Unknown                                24
Unsaturated hydrocarbons                5
Benzene and substituted derivatives     5
Lactones                                5
Steroids and steroid derivatives        4
Name: count, dtype: int64


In [ ]:
#imports
import os, re, time, json
import pandas as pd
import requests

#config
API_KEY = "gUhAcqhz.xaECZI2oJSFUS8NQ9BE4Z2Hp5hBJbfZg"
BASE = "http://35.184.189.38/api"
SLEEP_SEC = 0.15
CACHE_PATH = "hmdb_process_cache.json"

#load hmdb ids from intensity table
B = pd.read_csv("intensity_by_group.csv")
hmdb_ids = B["hmdb_id"].astype(str).str.strip().str.upper().unique().tolist()
print("HMDB IDs:", len(hmdb_ids))

#initialize cache
if os.path.exists(CACHE_PATH):
  with open(CACHE_PATH, "r") as f:
    cache = json.load(f)
else:
  cache = {}

#fetch hmdb ontology process JSON for metabolite
def fetch_process(hmdb_id: str):
  if hmdb_id in cache:
    return cache[hmdb_id]

  url = f"{BASE}/hmdb/metabolites/{hmdb_id}/ontology/process/"
  r = requests.get(url, params={"api-key": API_KEY}, timeout=30)
  data = r.json()

  cache[hmdb_id] = data
  time.sleep(SLEEP_SEC)
  return data

#collect human-readable ontology labels from json file
def extract_terms(obj):
  terms = []
  def walk(x):
    if isinstance(x, dict):
      for k, v in x.items():
        # common label-ish keys
        if k.lower() in {"name","label","term","value","ontology_term","description"} and isinstance(v, str):
          terms.append(v)
        walk(v)
    elif isinstance(x, list):
      for item in x:
        walk(item)
    elif isinstance(x, str):
      # sometimes the API returns a list of strings directly
      terms.append(x)
  walk(obj)

  # clean + dedupe
  cleaned = []
  for t in terms:
    t2 = t.strip()
    # drop trivial/non-informative very short tokens
    if t2 and len(t2) >= 4:
      cleaned.append(t2)

  # dedupe while preserving order
  seen = set()
  out = []
  for t in cleaned:
    if t not in seen:
      seen.add(t)
      out.append(t)
  return out

#map process terms to pathway categories (these come from HMDB)
CATEGORY_RULES = [
  ("Lipid metabolism", [r"\blipid\b", r"\bfatty acid\b", r"\bglycerolipid\b", r"\bphospholipid\b", r"\bsteroid\b", r"\bcholesterol\b", r"\bacylcarnitine\b"]),
  ("Amino acid metabolism", [r"\bamino acid\b", r"\bpeptide\b", r"\bprotein\b", r"\burea cycle\b", r"\bglutamate\b"]),
  ("Carbohydrate metabolism", [r"\bcarbohydrate\b", r"\bglycolysis\b", r"\bgluconeogenesis\b", r"\bglucose\b", r"\bfructose\b", r"\bgalactose\b", r"\bglycogen\b"]),
  ("Nucleotide metabolism", [r"\bnucleotide\b", r"\bnucleoside\b", r"\bpurine\b", r"\bpyrimidine\b", r"\bdna\b", r"\brna\b"]),
  ("Energy metabolism", [r"\btca\b", r"\bkrebs\b", r"\bcitric acid cycle\b", r"\boxidative phosphorylation\b", r"\bmitochond", r"\benergy\b", r"\batp\b"]),
  ("Cofactors and vitamins", [r"\bvitamin\b", r"\bcofactor\b", r"\bfolate\b", r"\bniacin\b", r"\briboflavin\b", r"\bthiamine\b", r"\bpyridox", r"\bbiotin\b"]),
  ("Xenobiotics and detox", [r"\bxenobiotic\b", r"\bdrug\b", r"\bdetox\b", r"\bcytochrome\b", r"\bglucuronidation\b", r"\bsulfation\b"]),
  ("Inflammation and immune", [r"\binflamm", r"\bimmune\b", r"\bcytokine\b", r"\bcomplement\b"]),
  ("Neurotransmitters and signaling", [r"\bneurotrans", r"\bseroton", r"\bdopamin", r"\bgaba\b", r"\bglutamate\b", r"\bsignaling\b", r"\breceptor\b"]),
]

#convert term list into one searchable text blob
def categorize(terms):
  txt = " | ".join([t.lower() for t in terms])
  for cat, patterns in CATEGORY_RULES:
    for p in patterns:
      if re.search(p, txt, flags=re.I):
        return cat
    return "Other/Unknown"

#buil hmdb to pathway catgeory table
rows = []
errors = 0

for i, hmdb_id in enumerate(hmdb_ids, 1):
  data = fetch_process(hmdb_id)
  terms = extract_terms(data)

  cat = categorize(terms)

  # store top few terms for traceability (optional)
  rows.append({
    "hmdb_id": hmdb_id,
    "pathway_category": cat,
    "process_terms_top": "; ".join(terms[:12])  # keep short
  })

  if i % 25 == 0:
    print(f"Processed {i}/{len(hmdb_ids)}")

# persist cache for future runs
with open(CACHE_PATH, "w") as f:
  json.dump(cache, f)

df_map = pd.DataFrame(rows)
print("Errors:", errors)
print(df_map["pathway_category"].value_counts())

#assign roman numerals to categories
def to_roman(n):
  vals = [(1000,'M'),(900,'CM'),(500,'D'),(400,'CD'),
          (100,'C'),(90,'XC'),(50,'L'),(40,'XL'),
          (10,'X'),(9,'IX'),(5,'V'),(4,'IV'),(1,'I')]
  out = []
  for v,sym in vals:
    while n >= v:
      out.append(sym); n -= v
  return "".join(out)

cats = df_map["pathway_category"].value_counts().index.tolist()
roman_map = {cat: to_roman(i+1) for i, cat in enumerate(cats)}

df_map["pathway_id"] = df_map["pathway_category"].map(roman_map)
df_map["pathway_name"] = df_map["pathway_category"]

#write hmdb_to_pathway.csv and chords.csv for circos plot
hmdb_to_pathway = df_map[["hmdb_id","pathway_id","pathway_name","pathway_category"]].copy()
hmdb_to_pathway.to_csv("hmdb_to_pathway.csv", index=False)

chords = hmdb_to_pathway[["hmdb_id","pathway_id"]].rename(columns={"hmdb_id":"from","pathway_id":"to"})
chords["weight"] = 1
chords.to_csv("chords.csv", index=False)

# Also write a legend table for use in R
legend = (hmdb_to_pathway[["pathway_id","pathway_name"]].drop_duplicates().sort_values("pathway_id"))
legend.to_csv("roman_legend.csv", index=False)

print("Wrote: hmdb_to_pathway.csv, chords.csv, roman_legend.csv")

HMDB IDs: 312
Processed 25/312
Processed 50/312
Processed 75/312
Processed 100/312
Processed 125/312
Processed 150/312
Processed 175/312
Processed 200/312
Processed 225/312
Processed 250/312
Processed 275/312
Processed 300/312
Errors: 0
pathway_category
Other/Unknown                      152
Lipid metabolism                   127
Carbohydrate metabolism             27
Neurotransmitters and signaling      6
Name: count, dtype: int64
Wrote: hmdb_to_pathway.csv, chords.csv, roman_legend.csv


In [ ]:
p = pd.read_csv("/content/hmdb_to_pathway.csv")
p

,hmdb_id,pathway_id,pathway_name,pathway_category
0,HMDB0059726,I,Other/Unknown,Other/Unknown
1,HMDB0003375,I,Other/Unknown,Other/Unknown
2,HMDB0036076,I,Other/Unknown,Other/Unknown
3,HMDB0034697,II,Lipid metabolism,Lipid metabolism
4,HMDB0006525,II,Lipid metabolism,Lipid metabolism
...,...,...,...,...
307,HMDB0038150,II,Lipid metabolism,Lipid metabolism
308,HMDB0010729,II,Lipid metabolism,Lipid metabolism
309,HMDB0030089,II,Lipid metabolism,Lipid metabolism
310,HMDB0037742,I,Other/Unknown,Other/Unknown


In [ ]:
#analyze lipid class counts
lipids = tax[tax["class"].str.contains("lipid", case=False, na=False)]
print(lipids["subclass"].value_counts().head(20))

subclass
Diradylglycerols               73
Monoterpenoids                 39
Glycerophosphates              32
Triradylcglycerols             18
CDP-glycerols                   4
Sesquiterpenoids                3
Glycerophosphoethanolamines     2
Diterpenoids                    1
Glycerophosphocholines          1
Terpene glycosides              1
Triterpenoids                   1
Name: count, dtype: int64


In [ ]:
B = B.copy()
B = B.rename(columns={"metabolite": "hmdb_id"})   # hmdb_id now exists
B = B.drop(columns=["metbolite"], errors="ignore")  # drop typo column

#make sure names are normalized
def norm_hmdb(s):
  s = s.astype(str).str.strip().str.upper()
  return s.str.extract(r"(HMDB\d+)", expand=False)

B["hmdb_id"] = norm_hmdb(B["hmdb_id"])

print(B.columns.tolist())
print(B.head(3))

['hmdb_id', 'Control', 'Depressed', 'delta']
       hmdb_id   Control  Depressed     delta
0  HMDB0059726  0.035375  -0.045958 -0.081333
1  HMDB0003375  0.035375  -0.045958 -0.081333
2  HMDB0036076  0.035375  -0.045958 -0.081333


In [ ]:
#merge intensity table with taxonomy annotations
df = B.merge(tax[["hmdb_id","superclass","class","subclass"]], on="hmdb_id", how="left")
print(df.shape)
print(df[["hmdb_id","class","subclass"]].head(10))
print("Missing class:", df["class"].isna().sum())

(313, 7)
       hmdb_id          class        subclass
0  HMDB0059726  Prenol lipids  Monoterpenoids
1  HMDB0003375  Prenol lipids  Monoterpenoids
2  HMDB0036076  Prenol lipids  Monoterpenoids
3  HMDB0034697  Prenol lipids  Monoterpenoids
4  HMDB0006525  Prenol lipids  Monoterpenoids
5  HMDB0041634  Prenol lipids  Monoterpenoids
6  HMDB0036116  Prenol lipids  Monoterpenoids
7  HMDB0035625  Prenol lipids  Monoterpenoids
8  HMDB0004321  Prenol lipids  Monoterpenoids
9  HMDB0034928  Prenol lipids  Monoterpenoids
Missing class: 24


In [ ]:
#analyze class and subclasses
print(df["class"].value_counts().head(15))
print(df["subclass"].value_counts().head(20))

class
Glycerolipids                          91
Prenol lipids                          45
Fatty Acyls                            44
Glycerophospholipids                   39
Carboxylic acids and derivatives       26
Lactones                                5
Benzene and substituted derivatives     5
Unsaturated hydrocarbons                5
Steroids and steroid derivatives        4
Organooxygen compounds                  3
Organonitrogen compounds                2
Indoles and derivatives                 2
Cinnamic acids and derivatives          2
Piperidines                             2
Stilbenes                               1
Name: count, dtype: int64
subclass
Diradylglycerols                        73
Monoterpenoids                          39
Fatty acid esters                       33
Glycerophosphates                       32
Amino acids, peptides, and analogues    21
Triradylcglycerols                      18
Branched unsaturated hydrocarbons        5
Fatty alcohol esters        

In [ ]:
df = df.copy()
#drop unknowns in taxonomy
df["class"] = df["class"].fillna("Unknown")
df["subclass"] = df["subclass"].fillna("Unknown")

df = df[(df["class"] != "Unknown") & (df["subclass"] != "Unknown")].copy()
print("Remaining metabolites after dropping Unknown:", df.shape[0])

#collapse rare subclasses into their parent class
MIN_N = 8
vc = df["subclass"].value_counts()
keep = set(vc[vc >= MIN_N].index)

df["pathway_label"] = df.apply(lambda r: r["subclass"] if r["subclass"] in keep else r["class"], axis=1)

print("Number of sectors:", df["pathway_label"].nunique())
print(df["pathway_label"].value_counts().head(25))

#Roman numerals
def to_roman(n):
  vals=[(1000,'M'),(900,'CM'),(500,'D'),(400,'CD'),
        (100,'C'),(90,'XC'),(50,'L'),(40,'XL'),
        (10,'X'),(9,'IX'),(5,'V'),(4,'IV'),(1,'I')]
  out=""
  for v,s in vals:
    while n>=v:
      out+=s; n-=v
  return out

cats = df["pathway_label"].value_counts().index.tolist()
roman_map = {cat: to_roman(i+1) for i,cat in enumerate(cats)}

df["pathway_id"] = df["pathway_label"].map(roman_map)
df["pathway_name"] = df["pathway_label"]

#write files for R
hmdb_to_pathway = df[["hmdb_id","pathway_id","pathway_name"]].drop_duplicates().copy()
hmdb_to_pathway["pathway_category"] = hmdb_to_pathway["pathway_name"]
hmdb_to_pathway.to_csv("hmdb_to_pathway.csv", index=False)

chords = hmdb_to_pathway[["hmdb_id","pathway_id"]].rename(columns={"hmdb_id":"from","pathway_id":"to"})
chords["weight"] = 1
chords.to_csv("chords.csv", index=False)

legend = hmdb_to_pathway[["pathway_id","pathway_name"]].drop_duplicates().sort_values("pathway_id")
legend.to_csv("roman_legend.csv", index=False)

print("\nLegend preview:")
print(legend.head(30))
print("\nWrote: hmdb_to_pathway.csv, chords.csv, roman_legend.csv")

Remaining metabolites after dropping Unknown: 282
Number of sectors: 28
pathway_label
Diradylglycerols                        73
Monoterpenoids                          39
Fatty acid esters                       33
Glycerophosphates                       32
Amino acids, peptides, and analogues    21
Triradylcglycerols                      18
Fatty Acyls                             11
Glycerophospholipids                     7
Prenol lipids                            6
Unsaturated hydrocarbons                 5
Benzene and substituted derivatives      5
Carboxylic acids and derivatives         5
Lactones                                 4
Steroids and steroid derivatives         4
Organooxygen compounds                   3
Organonitrogen compounds                 2
Indoles and derivatives                  2
Cinnamic acids and derivatives           2
Phenol ethers                            1
Dioxolanes                               1
Halohydrins                              1
Tannins    

In [ ]:
df2 = df.copy()

# drop unknowns again (safety)
df2 = df2[(df2["class"]!="Unknown") & (df2["subclass"]!="Unknown")].copy()

# keep only meaningful subclasses
MAJOR_SUBCLASSES = {
  "Diradylglycerols",
  "Glycerophosphates",
  "Fatty acid esters",
  "Triradylcglycerols",
  "Monoterpenoids",
  "Amino acids, peptides, and analogues"
}

def pick_sector(row):
    sub = row["subclass"]
    cls = row["class"]

    if sub in MAJOR_SUBCLASSES:
      return sub

    # roll lipids into their lipid class
    if "lipid" in str(cls).lower():
      return cls

    return "Non-lipid metabolites"

df2["pathway_label"] = df2.apply(pick_sector, axis=1)

print(df2["pathway_label"].value_counts())
print("Number of sectors:", df2["pathway_label"].nunique())

pathway_label
Diradylglycerols                        73
Non-lipid metabolites                   53
Monoterpenoids                          39
Fatty acid esters                       33
Glycerophosphates                       32
Amino acids, peptides, and analogues    21
Triradylcglycerols                      18
Glycerophospholipids                     7
Prenol lipids                            6
Name: count, dtype: int64
Number of sectors: 9


In [ ]:
#assign roman numerals
def to_roman(n):
  vals=[(10,'X'),(9,'IX'),(5,'V'),(4,'IV'),(1,'I')]
  out=""
  for v,s in vals:
    while n>=v:
      out+=s; n-=v
  return out

#rank pathway categories by frequency
cats = df2["pathway_label"].value_counts().index.tolist()
roman_map = {cat: to_roman(i+1) for i,cat in enumerate(cats)}

#map roman numeral ids to dataframe
df2["pathway_id"] = df2["pathway_label"].map(roman_map)

#build hmdb to pathway mapping table
hmdb_to_pathway = df2[["hmdb_id","pathway_id","pathway_label"]].drop_duplicates()
hmdb_to_pathway = hmdb_to_pathway.rename(columns={"pathway_label":"pathway_name"})

#write main mapping table
hmdb_to_pathway.to_csv("hmdb_to_pathway.csv", index=False)

#create chord ege list for circos plot
chords = hmdb_to_pathway[["hmdb_id","pathway_id"]].rename(columns={"hmdb_id":"from","pathway_id":"to"})
chords["weight"]=1
chords.to_csv("chords.csv", index=False)

#create legend table with roman numeral to pathway name
legend = hmdb_to_pathway[["pathway_id","pathway_name"]].drop_duplicates().sort_values("pathway_id")
legend.to_csv("roman_legend.csv", index=False)

print(legend)

    pathway_id                          pathway_name
107          I                      Diradylglycerols
17          II                 Non-lipid metabolites
0          III                        Monoterpenoids
23          IV                     Fatty acid esters
58          IX                         Prenol lipids
225          V                     Glycerophosphates
45          VI  Amino acids, peptides, and analogues
268        VII                    Triradylcglycerols
24        VIII                  Glycerophospholipids


In [ ]:
hmdb_to_pathway = pd.read_csv("hmdb_to_pathway.csv")  # columns: hmdb_id, pathway_id, pathway_name, ...

#Order pathway names by frequency (largest sector gets I)
order = (hmdb_to_pathway["pathway_name"].value_counts().index.tolist())

#convert integers to roman numerals
def to_roman(n: int) -> str:
  vals = [(1000,'M'),(900,'CM'),(500,'D'),(400,'CD'),
          (100,'C'),(90,'XC'),(50,'L'),(40,'XL'),
          (10,'X'),(9,'IX'),(5,'V'),(4,'IV'),(1,'I')]
  out = []
  for v,sym in vals:
    while n >= v:
      out.append(sym)
      n -= v
  return "".join(out)

roman_map = {name: to_roman(i+1) for i, name in enumerate(order)}

# overwrite pathway_id deterministically
hmdb_to_pathway["pathway_id"] = hmdb_to_pathway["pathway_name"].map(roman_map)

# save mapping
hmdb_to_pathway.to_csv("hmdb_to_pathway.csv", index=False)

# regenerate chords
chords = hmdb_to_pathway[["hmdb_id","pathway_id"]].rename(columns={"hmdb_id":"from","pathway_id":"to"})
chords["weight"] = 1
chords.to_csv("chords.csv", index=False)

# create legend sorted by Roman numeral NUMERIC value
roman_to_int_map = {"I":1,"V":5,"X":10,"L":50,"C":100,"D":500,"M":1000}
def roman_to_int(s: str) -> int:
  s = str(s).strip().upper()
  total, prev = 0, 0
  for ch in reversed(s):
    val = roman_to_int_map.get(ch, 0)
    if val < prev:
      total -= val
    else:
      total += val
      prev = val
  return total

legend = (hmdb_to_pathway[["pathway_id","pathway_name"]].drop_duplicates().assign(_n=lambda d: d["pathway_id"].map(roman_to_int)).sort_values("_n").drop(columns="_n").reset_index(drop=True))
legend.to_csv("roman_legend.csv", index=False)

print(legend)

  pathway_id                          pathway_name
0          I                      Diradylglycerols
1         II                 Non-lipid metabolites
2        III                        Monoterpenoids
3         IV                     Glycerophosphates
4          V                     Fatty acid esters
5         VI  Amino acids, peptides, and analogues
6        VII                    Triradylcglycerols
7       VIII                  Glycerophospholipids
8         IX                         Prenol lipids
